# Module 1 Data pipeline

Student starter notebook. Run cells in order. TODO sections are unfinished
assignment work; setup success is not a completed capstone.

In [ ]:
from pathlib import Path
import sys
from importlib.metadata import version, PackageNotFoundError

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "SUBMISSION_CHECKLIST.md").exists())
print("Python:", sys.version.split()[0])
print("Kernel:", Path(sys.executable).parent.name)
print("Project:", ROOT.name)

def show_versions(packages):
    for package in packages:
        try:
            print(f"{package}: {version(package)}")
        except PackageNotFoundError:
            print(f"{package}: missing (needed for full implementation)")

In [ ]:
show_versions(["requests", "beautifulsoup4", "pandas"])
import sqlite3
import pandas as pd

MODULE = ROOT / "data_pipeline"
BASE_URL = "https://books.toscrape.com/"
GBP_TO_INR = 105.50
print("Required fixed GBP to INR rate:", GBP_TO_INR)

## 1 Scrape real book data
TODO: Use requests with a timeout and raise_for_status, BeautifulSoup, and pagination.
Capture title, price, star_rating, availability, category for >=60 books across >=3 categories.
Store the resulting rows in a DataFrame named raw_books. Do not use fabricated data.

In [ ]:
# TODO: Fetch category/listing/detail pages and populate raw_books.

## 2 Clean and convert
Starter function: drop invalid required fields. Report and justify the actual dropped-row count.

In [ ]:
def clean_books(raw_books):
    books = raw_books.copy()
    books["price_gbp"] = pd.to_numeric(
        books["price"].astype(str).str.replace("£", "", regex=False).str.strip(),
        errors="coerce")
    books["rating"] = books["star_rating"].map(
        {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5})
    availability = books["availability"].astype(str).str.lower().str.strip()
    books["in_stock"] = availability.map(
        lambda value: True if value.startswith("in stock") else
        False if value.startswith("out of stock") else None)
    for column in ["title", "category"]:
        books[column] = books[column].astype("string").str.strip().replace("", pd.NA)
    books = books.dropna(subset=["title", "category", "price_gbp", "rating", "in_stock"])
    books = books.loc[books["price_gbp"] >= 0].copy()
    books["price_gbp"] = books["price_gbp"].astype(float)
    books["rating"] = books["rating"].astype(int)
    books["in_stock"] = books["in_stock"].astype(bool)
    books["price_inr"] = (books["price_gbp"] * GBP_TO_INR).round(2)
    print("Rows removed:", len(raw_books) - len(books))
    return books

print("Cleaning function ready; call clean_books(raw_books) after scraping.")

## 3 SQLite schema
The following checks a real schema in memory. TODO: load real records into data/books.db, preserving primary/foreign keys and making reruns safe.

In [ ]:
SCHEMA = """
PRAGMA foreign_keys = ON;
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL UNIQUE
);
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    in_stock INTEGER NOT NULL CHECK (in_stock IN (0, 1)),
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
);
"""
with sqlite3.connect(":memory:") as connection:
    connection.executescript(SCHEMA)
    print(connection.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())

## 4 SQL and pandas outputs
TODO: Execute >=5 queries including every required clause and a JOIN. Save actual query text and outputs.
Use pd.read_sql for >=2 queries, then pd.merge for the identical JOIN and assert equality after sorting.

In [ ]:
# TODO: Execute queries on populated data; show SQL and pandas JOIN results side by side.

## 5 My interpretation
TODO: Explain parsing decisions, data counts, schema and what each query shows. Update the module README.